# [WIP] Quality Controlling a CTD profile
Quality control of a shipboard CTD profile (Temperature & Salinity)

## Objective:
Walk throught the QC process using CoTeDe.

In [1]:
from bokeh.io import output_notebook, show
from bokeh.layouts import row
from bokeh.plotting import figure
import numpy as np

import cotede
from cotede import datasets, qctests

matplotlib is not available


In [2]:
output_notebook()

Loading BokehJS ...

## Data
We'll use a CTD profile in the Tropical Atlantic for this tutorial.
If curious about this dataset, check [CoTeDe's documentation](https://cotede.readthedocs.io) for more details.

Let's load the data and check which variables are available.

In [3]:
data = cotede.datasets.load_ctd()

print("The variables are: ", ", ".join(sorted(data.keys())))
print("There is a total of {} observed depths.".format(len(data["TEMP"])))

The variables are:  CNDC, CNDC2, PRES, PSAL, PSAL2, TEMP, TEMP2, flag, potemperature, potemperature2, timeS
There is a total of 1014 observed depths.


This CTD was equipped with backup sensors to provide more robustness.
Measurements from the secondary sensor are identified by a 2 in the end of the name. For instance, TEMP2 is the secondary temperature sensor.
Here, we will focus on the primary sensors.

To visualize this profile we will use Bokeh which allows to make interactive plots.

In [4]:
p1 = figure(width=420, height=600)
p1.scatter(data['TEMP'], -data['PRES'],
          size=8, line_color="seagreen", fill_color="mediumseagreen", fill_alpha=0.3)
p1.xaxis.axis_label = "Temperature [C]"
p1.yaxis.axis_label = "Depth [m]"

p2 = figure(width=420, height=600)
p2.y_range = p1.y_range
p2.scatter(data['PSAL'], -data['PRES'],
          size=8, line_color="seagreen", fill_color="mediumseagreen", fill_alpha=0.3)
p2.xaxis.axis_label = "Salinity"
p2.yaxis.axis_label = "Depth [m]"

p = row(p1, p2)
show(p)

Considering the unusual magnitudes and variability near the bottom, there are clearly bad measurements in this profile.
Let's start with one of the most fundamental QC test and restrict the profile to feasible values.

## Global Range: Check for Feasible Values
Let's use the thresholds recommended by the [GTSPP](https://cotede.readthedocs.io/en/latest/qctests.html):
 - Temperature between -2 and 40 $^\circ$C
 - Salinity between 0 and 41

In [5]:
# ToDo: Include a shaded area for unfeasible values

idx_valid = (data['TEMP'] > -2) & (data['TEMP'] < 40)

p1 = figure(width=420, height=600, title="Global Range Check (-2 <= T <= 40)")
p1.scatter(data['TEMP'][idx_valid], -data['PRES'][idx_valid], size=8, line_color="seagreen", fill_color="mediumseagreen", fill_alpha=0.3, legend_label="Good values")
p1.scatter(data['TEMP'][~idx_valid], -data['PRES'][~idx_valid], size=8, line_color="red", fill_color="red", fill_alpha=0.3, legend_label="Bad values", marker='triangle')
p1.xaxis.axis_label = "Temperature [C]"
p1.yaxis.axis_label = "Depth [m]"


idx_valid = (data['PSAL'] > 0) & (data['PSAL'] < 41)

p2 = figure(width=420, height=600, title="Global Range Check (0 <= S <= 41)")
p2.y_range = p1.y_range
p2.scatter(data['PSAL'][idx_valid], -data['PRES'][idx_valid], size=8, line_color="seagreen", fill_color="mediumseagreen", fill_alpha=0.3, legend_label="Good values")
p2.scatter(data['PSAL'][~idx_valid], -data['PRES'][~idx_valid], size=8, line_color="red", fill_color="red", fill_alpha=0.3, legend_label="Bad values", marker='triangle')
p2.xaxis.axis_label = "Pratical Salinity"
p2.yaxis.axis_label = "Depth [m]"

p = row(p1, p2)
show(p)

Great, we already identified a fair number of bad measurements.
The global range test is a simple and light test, and there is no reason to always apply it in normal conditions, but it is usually not enough.
We will need to apply more tests to capture the rest of the bad measurements.

Several QC tests were already implemented in CoTeDe, so you don't need to code it again.
For instance, the global range test is available as `qctests.GlobalRange` and we can use it like

In [6]:
y = qctests.GlobalRange(data, varname='TEMP', cfg={"minval": -2, "maxval": 40})
y.flags

{'global_range': array([1, 1, 1, ..., 4, 1, 4], shape=(1014,), dtype=int8)}

Let's use that to check what are the unfeasible values of temperature.

In [7]:
flag = y.flags["global_range"]
data["TEMP"][flag==4]

array([-14.70619965,  -5.89820004, -20.52770042, -12.84549999,
        -7.79710007, -22.98450089,  48.20959854,  98.05940247,
        53.17100143,  -7.30980015, -17.39480019,  48.66450119,
        98.45189667, -19.34189987])

The Global Range is a trivial one to implement, but there are other checks that are more complex and CoTeDe provides a solution for that.
For instance, let's consider another traditional procedure, the Spike check.

## Spike
The spike check is a quite traditional one and is based on the principle of comparing one measurement with the tendency observed from the neighbor values.
We could implement it as follows:

In [8]:
def spike(x):
    """Spike check as defined by GTSPP
    
    Notes
    -----
    - Check CoTeDe's manual for more details.
    """
    y = np.nan * x
    y[1:-1] = np.abs(x[1:-1] - (x[:-2] + x[2:]) / 2.0) - np.abs((x[2:] - x[:-2]) / 2.0)
    return y

This is already implemented in CoTeDe as `qctests.spike`, and we could use it as shown below:

In [9]:
temp_spike = qctests.spike(data["TEMP"])

print("The largest spike observed was: {:.3f}".format(np.nanmax(np.abs(temp_spike))))

The largest spike observed was: 58.529


The same could be done for salinity, such as: ``sal_spike = qctests.spike(data["PSAL"])``

The traditional approach to use the spike check is by comparing the "spikeness magnitude" with a threshold.
The measurement is considered bad (flag 4) if the spike is larger than that threshold.
Similar to the global range check, we could hence use the `spike()` and compare the output with acceptable limits.
This procedure is already available in CoTeDe as `qctests.Spike` and we can use it as follows,

In [10]:
y_spike = qctests.Spike(data, "TEMP", cfg={"threshold": 2.0})
y_spike.flags

{'spike': array([0, 1, 1, ..., 4, 4, 0], shape=(1014,), dtype=int8)}

Like the Global Range, it provides the quality flags obtained from this procedure.
Note that the standard flagging follows the IOC recommendation, thus 1 means good data while 0 is no QC applied.
To customize the flags, check the manual for custom configuration.
The spike check is based on the previous and following measurements, thus it can't evaluate the first nor the last values, returning flag 0 for those two measurements.

Some procedures provide more than just the flags, but also include features derived from the original measurements.
For instance, if one was interested in the "spike intensity" of one measurement, that could be inspected as:

In [11]:
y_spike.features

{'spike': array([            nan, -4.49943542e-03,  6.99996948e-04, ...,
         4.97873955e+01, -5.85286980e+01,             nan], shape=(1014,))}

The magnitudes of the tests are stored in `features`.

## More tests
QC checks are usually focused on specific characteristics of bad measurements, thus to cover a wider range of issues we typically combine a set of checks.
Let's apply the Gradient and the Tukey53H checks

In [12]:
y_gradient = qctests.Gradient(data, "TEMP", cfg={"threshold": 10})
y_gradient.flags

{'gradient': array([0, 1, 1, ..., 4, 1, 0], shape=(1014,), dtype=int8)}

In [13]:
y_tukey53H = qctests.Tukey53H(data, "TEMP", cfg={"threshold": 2.0})
y_tukey53H.flags

{'tukey53H': array([0, 0, 0, ..., 0, 0, 0], shape=(1014,), dtype=int8)}

These already implemented tests are useful, but it could be easier.
We usually don't apply one test at a time but a set of tests.
We could do that by defining a QC configuration like

In [14]:
cfg = {
    "TEMP": {
        "global_range": {"minval": -2, "maxval": 40},
        "gradient": {"threshold": 10.0},
        "spike": {"threshold": 2.0},
        "tukey53H": {"threshold": 1.5},
    },
    "PSAL": {
        "global_range": {"minval": 0, "maxval": 41},
        "gradient": {"threshold": 5.0},
        "spike": {"threshold": 0.3},
        "tukey53H": {"threshold": 1.0},
    }
}

In [15]:
pqc = cotede.ProfileQC(data, cfg=cfg)

That's it, the temperature and salinity from the primary sensor were evaluated.
Let's explore this pqc object.

The same variables in the input are available in the output object.

In [16]:
print("Variables available in data: {}\n".format(", ".join(data.keys())))
print("Variables available in pqc: {}\n".format(", ".join(pqc.keys())))

Variables available in data: timeS, PRES, TEMP, TEMP2, CNDC, CNDC2, potemperature, potemperature2, PSAL, PSAL2, flag

Variables available in pqc: timeS, PRES, TEMP, TEMP2, CNDC, CNDC2, potemperature, potemperature2, PSAL, PSAL2, flag



But only the variables in the `cfg` dictionary were QC'd

In [17]:
print("Variables flagged in pqc: {}\n".format(", ".join(pqc.flags.keys())))

Variables flagged in pqc: TEMP, PSAL



In [18]:
print("Flags available for temperature {}\n".format(pqc.flags["TEMP"].keys()))
print("Flags available for salinity {}\n".format(pqc.flags["PSAL"].keys()))

Flags available for temperature dict_keys(['global_range', 'gradient', 'spike', 'tukey53H', 'overall'])

Flags available for salinity dict_keys(['global_range', 'gradient', 'spike', 'tukey53H', 'overall'])



In [19]:
flag = pqc.flags["TEMP"]["overall"]
print('Overall flags for TEMP:', flag)

Overall flags for TEMP: [1 1 1 ... 4 4 4]


The flags are on IOC standard, thus 1 means good while 4 means bad.
0 is used when the QC there was no QC. For instance, the spike test is defined so that it depends on the previous and following measurements, thus the first and last data point of the array will always have a spike flag equal to 0.

## Using CoTeDe QC framework
CoTeDe automates many procedures for QC. Let's start using the standard procedure.

That's it, the primary and secondary sensors were evaluated. First the same variables in the input are available in the output object.

In [20]:
print("Variables available in data: {}\n".format(data.keys()))
print("Variables available in pqc: {}\n".format(pqc.keys()))

Variables available in data: dict_keys([np.str_('timeS'), np.str_('PRES'), np.str_('TEMP'), np.str_('TEMP2'), np.str_('CNDC'), np.str_('CNDC2'), np.str_('potemperature'), np.str_('potemperature2'), np.str_('PSAL'), np.str_('PSAL2'), np.str_('flag')])

Variables available in pqc: dict_keys([np.str_('timeS'), np.str_('PRES'), np.str_('TEMP'), np.str_('TEMP2'), np.str_('CNDC'), np.str_('CNDC2'), np.str_('potemperature'), np.str_('potemperature2'), np.str_('PSAL'), np.str_('PSAL2'), np.str_('flag')])



In [21]:
print("Flags available for temperature {}\n".format(pqc.flags["TEMP"].keys()))
print("Flags available for salinity {}\n".format(pqc.flags["PSAL"].keys()))

Flags available for temperature dict_keys(['global_range', 'gradient', 'spike', 'tukey53H', 'overall'])

Flags available for salinity dict_keys(['global_range', 'gradient', 'spike', 'tukey53H', 'overall'])



The flags are on IOC standard, thus 1 means good while 4 means bad.
0 is used when the QC there was no QC. For instance, the spike test is defined so that it depends on the previous and following measurements, thus the first and last data point of the array will always have a spike flag equal to 0.

Let's check the salinity with feasible values:

In [22]:
# ToDo: Include a shaded area for unfeasible values

idx_valid = (pqc.flags["TEMP"]["overall"] <= 2)

p1 = figure(width=420, height=600, title="Global Range Check (-2 <= T <= 40)")
p1.scatter(data['TEMP'][idx_valid], -data['PRES'][idx_valid], size=8, line_color="seagreen", fill_color="mediumseagreen", fill_alpha=0.3, legend_label="Good values")
p1.scatter(data['TEMP'][~idx_valid], -data['PRES'][~idx_valid], size=8, line_color="red", fill_color="red", fill_alpha=0.3, legend_label="Bad values", marker='triangle')
p1.xaxis.axis_label = "Temperature [C]"
p1.yaxis.axis_label = "Depth [m]"


idx_valid = (pqc.flags["PSAL"]["overall"] <= 2)

p2 = figure(width=420, height=600, title="Global Range Check (0 <= S <= 41)")
p2.y_range = p1.y_range
p2.scatter(data['PSAL'][idx_valid], -data['PRES'][idx_valid], size=8, line_color="seagreen", fill_color="mediumseagreen", fill_alpha=0.3, legend_label="Good values")
p2.scatter(data['PSAL'][~idx_valid], -data['PRES'][~idx_valid], size=8, line_color="red", fill_color="red", fill_alpha=0.3, legend_label="Bad values", marker='triangle')
p2.xaxis.axis_label = "Pratical Salinity"
p2.yaxis.axis_label = "Depth [m]"

p = row(p1, p2)
show(p)

## More tests: GTSPP Spike and Gradient tests
OK, let's apply more tests beyond the global range.
Some common ones are the gradient and spike, and we could use CoTeDe to run that like

In [23]:
y_gradient = qctests.Gradient(data, 'TEMP', cfg={"threshold": 10})
y_gradient.flags

{'gradient': array([0, 1, 1, ..., 4, 1, 0], shape=(1014,), dtype=int8)}

In [24]:
y_spike = qctests.Spike(data, 'TEMP', cfg={"threshold": 2.0})
y_spike.flags

{'spike': array([0, 1, 1, ..., 4, 4, 0], shape=(1014,), dtype=int8)}

## The Easiest Way: High level
Let's evaluate this profile using EuroGOOS standard tests.

In [25]:
pqced = cotede.ProfileQCed(data, cfg='eurogoos')

Using pressure as depth without any correction!
Using pressure as depth without any correction!


In [26]:
p = figure(width=500, height=600)
p.scatter(pqced['TEMP'], -pqced['PRES'], size=8, line_color="green", fill_color="green", fill_alpha=0.3)
show(p)

## QC with more control: "medium" level

In [27]:
pqc = cotede.ProfileQC(data, cfg='eurogoos')

Using pressure as depth without any correction!
Using pressure as depth without any correction!


In [28]:
pqc.keys()

dict_keys([np.str_('timeS'), np.str_('PRES'), np.str_('TEMP'), np.str_('TEMP2'), np.str_('CNDC'), np.str_('CNDC2'), np.str_('potemperature'), np.str_('potemperature2'), np.str_('PSAL'), np.str_('PSAL2'), np.str_('flag')])

In [29]:
pqc.flags["TEMP"]

{'valid_datetime': array([1, 1, 1, ..., 1, 1, 1], shape=(1014,), dtype=int8),
 'location_at_sea': array([1, 1, 1, ..., 1, 1, 1], shape=(1014,), dtype=int8),
 'global_range': array([1, 1, 1, ..., 4, 1, 4], shape=(1014,), dtype=int8),
 'digit_roll_over': array([0, 1, 1, ..., 4, 4, 4], shape=(1014,), dtype=int8),
 'gradient_depthconditional': array([0, 1, 1, ..., 4, 1, 0], shape=(1014,), dtype=int8),
 'spike_depthconditional': array([0, 1, 1, ..., 4, 1, 0], shape=(1014,), dtype=int8),
 'woa_normbias': array([1, 1, 1, ..., 3, 3, 3], shape=(1014,), dtype=int8),
 'overall': array([1, 1, 1, ..., 4, 4, 4], shape=(1014,), dtype=int8)}

In [30]:
data.keys()

dict_keys([np.str_('timeS'), np.str_('PRES'), np.str_('TEMP'), np.str_('TEMP2'), np.str_('CNDC'), np.str_('CNDC2'), np.str_('potemperature'), np.str_('potemperature2'), np.str_('PSAL'), np.str_('PSAL2'), np.str_('flag')])

### Low level

In [31]:
from cotede import qctests
y = qctests.GlobalRange(data, 'TEMP', cfg={'minval': -4, "maxval": 45 })
y.flags

{'global_range': array([1, 1, 1, ..., 4, 1, 4], shape=(1014,), dtype=int8)}

In [32]:
y = qctests.Tukey53H(data, 'TEMP', cfg={'threshold': 6, "l": 12})
y.features["tukey53H"]
p = figure(width=500, height=600)
p.scatter(y.features["tukey53H"], -data['PRES'], size=8, line_color="green", fill_color="green", fill_alpha=0.3)
show(p)

In [33]:
cfg = {'TEMP': {'global_range': {'minval': -4, 'maxval': 45}}}

pqc = cotede.ProfileQC(data, cfg)

pqc.flags['TEMP']
pqc.flags['TEMP']['overall']

idx_good = pqc.flags['TEMP']['overall'] <= 2
idx_bad = pqc.flags['TEMP']['overall'] >= 3

p = figure(width=500, height=600)
p.scatter(data['TEMP'][idx_good], -data['PRES'][idx_good], size=8, line_color="green", fill_color="green", fill_alpha=0.3)
p.scatter(data['TEMP'][idx_bad], -data['PRES'][idx_bad], size=8, line_color="red", fill_color="red", fill_alpha=0.3, marker='triangle')
show(p)

In [34]:
cfg['TEMP']['spike'] = {'threshold': 6}

pqc = cotede.ProfileQC(data, cfg)

pqc.flags['TEMP']
pqc.flags['TEMP']['overall']

idx_good = pqc.flags['TEMP']['overall'] <= 2
idx_bad = pqc.flags['TEMP']['overall'] >= 3

p = figure(width=500, height=600)
p.scatter(data['TEMP'][idx_good], -data['PRES'][idx_good], size=8, line_color="green", fill_color="green", fill_alpha=0.3)
p.scatter(data['TEMP'][idx_bad], -data['PRES'][idx_bad], size=8, line_color="red", fill_color="red", fill_alpha=0.3, marker='triangle')
show(p)

In [35]:
cfg['TEMP']['woa_normbias'] = {'threshold': 6}


pqc = cotede.ProfileQC(data, cfg)

pqc.flags['TEMP']
pqc.flags['TEMP']['overall']

idx_good = pqc.flags['TEMP']['overall'] <= 2
idx_bad = pqc.flags['TEMP']['overall'] >= 3

p = figure(width=500, height=600)
p.scatter(data['TEMP'][idx_good], -data['PRES'][idx_good], size=8, line_color="green", fill_color="green", fill_alpha=0.3)
p.scatter(data['TEMP'][idx_bad], -data['PRES'][idx_bad], size=8, line_color="red", fill_color="red", fill_alpha=0.3, marker='triangle')
show(p)

Using pressure as depth without any correction!


In [36]:
cfg['TEMP']['spike_depthconditional'] = {"pressure_threshold": 500, "shallow_max": 6.0, "deep_max": 2.0}

pqc = cotede.ProfileQC(data, cfg)

pqc.flags['TEMP']
pqc.flags['TEMP']['overall']

idx_good = pqc.flags['TEMP']['overall'] <= 2
idx_bad = pqc.flags['TEMP']['overall'] >= 3

p = figure(width=500, height=600)
p.scatter(data['TEMP'][idx_good], -data['PRES'][idx_good], size=8, line_color="green", fill_color="green", fill_alpha=0.3)
p.scatter(data['TEMP'][idx_bad], -data['PRES'][idx_bad], size=8, line_color="red", fill_color="red", fill_alpha=0.3, marker='triangle')
show(p)

Using pressure as depth without any correction!


In [37]:
## The Easiest Way: High level
# Let's evaluate this profile using EuroGOOS standard tests.

In [38]:
pqced = cotede.ProfileQCed(data, cfg='eurogoos')

Using pressure as depth without any correction!
Using pressure as depth without any correction!


In [39]:
p = figure(width=500, height=600)
p.scatter(pqced['TEMP'], -pqced['PRES'], size=8, line_color="green", fill_color="green", fill_alpha=0.3)
show(p)

In [40]:
## QC with more control: "medium" level

In [41]:
pqc = cotede.ProfileQC(data, cfg='eurogoos')

Using pressure as depth without any correction!
Using pressure as depth without any correction!


In [42]:
pqc.keys()

dict_keys([np.str_('timeS'), np.str_('PRES'), np.str_('TEMP'), np.str_('TEMP2'), np.str_('CNDC'), np.str_('CNDC2'), np.str_('potemperature'), np.str_('potemperature2'), np.str_('PSAL'), np.str_('PSAL2'), np.str_('flag')])

In [43]:
pqc.flags["TEMP"]

{'valid_datetime': array([1, 1, 1, ..., 1, 1, 1], shape=(1014,), dtype=int8),
 'location_at_sea': array([1, 1, 1, ..., 1, 1, 1], shape=(1014,), dtype=int8),
 'global_range': array([1, 1, 1, ..., 4, 1, 4], shape=(1014,), dtype=int8),
 'digit_roll_over': array([0, 1, 1, ..., 4, 4, 4], shape=(1014,), dtype=int8),
 'gradient_depthconditional': array([0, 1, 1, ..., 4, 1, 0], shape=(1014,), dtype=int8),
 'spike_depthconditional': array([0, 1, 1, ..., 4, 1, 0], shape=(1014,), dtype=int8),
 'woa_normbias': array([1, 1, 1, ..., 3, 3, 3], shape=(1014,), dtype=int8),
 'overall': array([1, 1, 1, ..., 4, 4, 4], shape=(1014,), dtype=int8)}

In [44]:
data.keys()

dict_keys([np.str_('timeS'), np.str_('PRES'), np.str_('TEMP'), np.str_('TEMP2'), np.str_('CNDC'), np.str_('CNDC2'), np.str_('potemperature'), np.str_('potemperature2'), np.str_('PSAL'), np.str_('PSAL2'), np.str_('flag')])

In [45]:
### Low level

In [46]:
from cotede import qctests
y = qctests.GlobalRange(data, 'TEMP', cfg={'minval': -4, "maxval": 45 })
y.flags

{'global_range': array([1, 1, 1, ..., 4, 1, 4], shape=(1014,), dtype=int8)}

In [47]:
y = qctests.Tukey53H(data, 'TEMP', cfg={'threshold': 6, "l": 12})
y.features["tukey53H"]
p = figure(width=500, height=600)
p.scatter(y.features["tukey53H"], -data['PRES'], size=8, line_color="green", fill_color="green", fill_alpha=0.3)
show(p)